# AEGEAN | Project integrity check
Run all cells from `Python Code/`. No scraping, inference, API requests, imports of project scripts or checkpoint deserialization. The audit checks the documented final project layout and distinguishes required production inputs from optional historical and regenerable outputs. Reports are written only to `Presentation/Integrity_Check/` when enabled. A pass is not an end-to-end execution test.

In [1]:
from pathlib import Path
from datetime import datetime
import ast, json, zipfile, csv, xml.etree.ElementTree as ET
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'Python Code').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Launch Jupyter inside the project or its Python Code folder.')
SAVE_REPORT = True
CHECK_CSV_CONTENT = True  # Read full CSVs in chunks; disable for a quick inventory.
EXPECTED_MATCHED = 30219
EXPECTED_PRESENTATION = 25568
print('Project root:', ROOT)

Project root: C:\Users\mfess\GitHub_Repos\aegean airlines


## 1. Expected files
Required items support the full production workflow and final deliverables. Optional items support recovery, historical experiments or regenerable diagnostics. Adjust the manifest if you intentionally retire a stage.

In [2]:
manifest = []
def add(stage, base, names, required=True):
    for name in names:
        manifest.append((stage, str(Path(base) / name) + ("/" if name.endswith("/") else ""), required))

add('Documentation', '.', ['README.md','PROJECT_README.md'])
add('Structure', '.', ['Python Code/','Data/','Data/Scraping/','Data/Output/','Data/Output/Cleaning data/','Data/Output/Hugging Face/','Data/Output/LLM/','Streamlit/','Presentation/','Python Code/Presentation/'])
add('Scraping code','Python Code/Scraping', ['Airline Ratings scraper.ipynb','eSky Scraper.ipynb','Flight Review Scraper.ipynb','Kayak Scraping.ipynb','TripAdvisor_scraper_apify.ipynb','Trustpilot - Apify Scraper.ipynb','airlineratings_multi_airline_paginated_v2.py'])
add('Scraped data','Data/Scraping', ['airlineratings_data/all_airlines_reviews.csv','esky/esky_all_reviews_v2.csv','flight_report_data/all_airlines_reviews.csv','kayak/kayak_all_airline_reviews.csv','tripadvisor_reviews/tripadvisor_all_airlines.csv','trustpilot_reviews/trustpilot_all_airlines.csv'])
add('Cleaning code','Python Code/Data Cleaning',['Airline_Reviews_Pipeline_Updated(1).ipynb'])
add('Legacy / incremental','Python Code/Data Cleaning',['Clean_Data(1).ipynb','Incremental_reviews_pipeline.ipynb'],False)
CLEAN='Data/Output/Cleaning data'; HF='Data/Output/Hugging Face'; LLM='Data/Output/LLM'
add('Cleaning data',CLEAN,['df_final.csv','airline_reviews_cleaned.csv','airline_reviews_translated.csv','airline_review_summary.xlsx'])
add('Translation recovery',CLEAN,['translation_checkpoint.pkl','incremental_translation_checkpoint.csv','Legacy/airline_translation_checkpoint.pkl'],False)
add('HF code','Python Code/Hugging Face',['Airline_Feature_Engineering_v2(1).ipynb','airline_features_v2.py','NoLLM_Sentiment_Analysis.ipynb'])
add('HF data',HF,['df_final_features.csv','df_final_absa.csv'])
add('HF diagnostics',HF,['df_final_features_no_first_class_no_long_haul.csv','df_final_absa_no_first_class_no_long_haul.csv','classification_diagnostics.csv','classification_conflicts.csv','classification_validation_sample.csv','feature_summary.csv','feature_summary_by_airline.csv','feature_summary_by_source.csv','absa_diagnostics.csv','absa_summary.csv','absa_airline_sentiment_summary.csv'],False)
add('Gemini code','Python Code/LLM',['LLM_sentiment_analysis.ipynb'])
add('Gemini data',LLM,['df_final_absa_gemini.csv'])
add('Gemini recovery / diagnostics',LLM,['df_final_absa_gemini_complete.csv','gemini_full_checkpoint.jsonl','gemini_full_failures.jsonl','gemini_diagnostics.csv','gemini_cost_summary.csv','gemini_vs_huggingface.csv','gemini_missing_reviews.csv','model_validation_100_reviews.xlsx'],False)
add('Pilot',LLM+'/Pilot',['df_final_absa_haiku.csv','df_final_absa_gemini.csv','haiku_checkpoint.jsonl','gemini_checkpoint.jsonl','haiku_failures.jsonl','gemini_failures.jsonl','llm_cost_comparison.csv','llm_feature_agreement.csv','llm_full_comparison.csv'],False)
add('Validation code','Python Code/Model Validation',['model_validation.ipynb'])
add('Human ground truth',LLM,['model_validation_100_reviewsv3.xlsx'])
add('Validation results',LLM+'/validation_results',['model_validation_results.xlsx','presentation_validation_summary.xlsx','summary.csv'])
add('Validation diagnostics',LLM+'/validation_results',['model_ranking.csv','feature_metrics.csv','class_metrics.csv','multilabel_metrics.csv','confidence_intervals.csv','model_differences.csv','model_agreement.csv','confusion_matrices.csv','overall_errors.csv','aspect_errors.csv','evaluation_coverage.csv','filter_audit.csv','overall_predictions.csv','aspect_predictions.csv','charts/'],False)
add('Streamlit','Streamlit',['app.py','README.md','requirements.txt','assets/linkedin_qr.svg'])
add('Presentation code','Python Code/Presentation',['generate_presentation_graphs.py'])
add('Presentation','Presentation',['Final Presentation.pptx'])
for stem in ['slide_08_sample','slide_09_validation','slide_10_drivers','slide_11_overall_benchmark','slide_12_best_peer','slide_13_best_peer','slide_14_evolution','slide_15_feature_changes']:
    add('Presentation graphs','Presentation',[stem+'.'+ext for ext in ['png','svg','csv']])
add('Repository hygiene','.', ['.gitignore','requirements.txt'],False)
print(f'{len(manifest)} expected paths')

122 expected paths


## 2. File existence and structural checks

In [3]:
results=[]
def report(stage,path,required,status,detail):
    results.append(dict(stage=stage,path=path,required=required,status=status,detail=detail))

def inspect_file(p):
    suffix=p.suffix.lower()
    if suffix=='.py': ast.parse(p.read_text(encoding='utf-8-sig'))
    elif suffix=='.ipynb':
        obj=json.loads(p.read_text(encoding='utf-8-sig'))
        if not isinstance(obj.get('cells'),list) or not obj['cells']:
            raise ValueError('No notebook cells')
    elif suffix in {'.xlsx','.pptx'}:
        with zipfile.ZipFile(p) as z:
            if z.testzip() is not None: raise ValueError('Corrupt ZIP member')
            expected='xl/workbook.xml' if suffix=='.xlsx' else 'ppt/presentation.xml'
            if expected not in z.namelist(): raise ValueError('Missing '+expected)
    elif suffix=='.svg': ET.parse(p)
    elif suffix=='.png':
        with p.open('rb') as f:
            if f.read(8)!=b'\x89PNG\r\n\x1a\n': raise ValueError('Invalid PNG signature')
    elif suffix=='.csv':
        if CHECK_CSV_CONTENT:
            count=sum(len(chunk) for chunk in pd.read_csv(p,chunksize=10000,low_memory=False))
        else: count=len(pd.read_csv(p,nrows=1))
        if not count: raise ValueError('CSV has no data rows')
        return f'{count:,} rows' if CHECK_CSV_CONTENT else 'Header and first row readable'
    elif suffix=='.jsonl':
        with p.open(encoding='utf-8-sig') as f:
            count=sum(1 for line in f if line.strip() and json.loads(line) is not None)
        return f'{count:,} JSON records'
    elif suffix=='.pkl': return 'Present; deliberately not deserialized'
    return 'Basic structural check passed'

for stage, rel, required in manifest:
    p=ROOT/rel
    try:
        if not p.exists(): report(stage,rel,required,'MISSING' if required else 'WARNING','Required path absent' if required else 'Optional path absent'); continue
        if rel.endswith('/'):
            if not p.is_dir(): raise ValueError('Expected directory')
            detail='Directory exists'
        else:
            if not p.is_file(): raise ValueError('Expected file')
            if p.stat().st_size==0:
                if not required and p.suffix=='.jsonl':
                    report(stage,rel,required,'WARNING','Empty optional log; may indicate no failures'); continue
                raise ValueError('Empty file')
            detail=inspect_file(p)
        report(stage,rel,required,'PASS',detail)
    except Exception as exc: report(stage,rel,required,'ERROR',str(exc))
print('Inventory complete.')

Inventory complete.


## 3. Production dataset compatibility and matched sample
Checks actual downstream column contracts, unique record keys, dates, airline coverage and the two different analytical scopes. Historical counts are reference values, not mandatory targets.

In [4]:
FEATURES='cancellation delay boarding baggage customer_service cabin_crew ground_staff complaint_resolution seat_comfort food_beverage cabin_cleanliness entertainment_wifi booking_experience check_in airport_experience value_for_money pricing_fees flight_disruption overall_experience'.split()
AIRLINES=['Aegean Airlines','Turkish Airlines','Air France','Lufthansa','Bangkok Air','Porter Airlines','Virgin Australia']
ALIASES={a.lower():a for a in AIRLINES} | {a.lower().replace(' ','_'):a for a in AIRLINES} | {'aegean':'Aegean Airlines','bangkok airways':'Bangkok Air','bangkok_airways':'Bangkok Air'}
VALID={'positive','negative','neutral','mixed'}
contracts={CLEAN+'/df_final.csv':['record_key','airline','review_date','source'], HF+'/df_final_features.csv':['record_key']+FEATURES, HF+'/df_final_absa.csv':['record_key','airline','review_date','source','original_language','long_haul','first_class','overall_sentiment']+[c for f in FEATURES for c in (f,f+'_sentiment')], LLM+'/df_final_absa_gemini.csv':['record_key','gemini_overall_sentiment']+[c for f in FEATURES for c in (f+'_llm',f+'_sentiment_llm')]}
loaded={}
for rel,cols in contracts.items():
    try:
        p=ROOT/rel
        header=pd.read_csv(p,nrows=0).columns
        missing=sorted(set(cols)-set(header))
        if missing: raise ValueError('Missing columns: '+', '.join(missing))
        df=pd.read_csv(p,usecols=cols,dtype={'record_key':'string'},low_memory=False)
        df['record_key']=df.record_key.str.strip()
        bad=int((df.record_key.isna() | df.record_key.eq('')).sum()); dup=int(df.record_key.duplicated().sum())
        if bad or dup: raise ValueError(f'{bad} blank keys; {dup} duplicate keys')
        if df.empty: raise ValueError('No records')
        loaded[rel]=df
        report('Dataset contract',rel,True,'PASS',f'{len(df):,} unique records; required columns present')
    except Exception as exc: report('Dataset contract',rel,True,'ERROR',str(exc))

sample_summary=pd.DataFrame()
def flag(s): return s.astype('string').str.strip().str.lower().isin(['1','1.0','true','yes','y'])
def check(name, condition, detail, required=False):
    report('Matched sample',name,required,'PASS' if condition else ('ERROR' if required else 'WARNING'),detail)
try:
    h=loaded[HF+'/df_final_absa.csv']; g=loaded[LLM+'/df_final_absa_gemini.csv']
    d=h.merge(g,on='record_key',validate='one_to_one')
    check('Nonempty intersection',not d.empty,f'{len(d):,} matched records',True)
    check('Historical matched count',len(d)==EXPECTED_MATCHED,f'Actual {len(d):,}; historical {EXPECTED_MATCHED:,}')
    check('Gemini keys in HF',g.record_key.isin(h.record_key).all(),f'{(~g.record_key.isin(h.record_key)).sum():,} Gemini-only keys')
    dates=pd.to_datetime(d.review_date,errors='coerce',utc=True)
    d['airline']=d.airline.astype('string').str.strip().str.lower().map(ALIASES)
    scope=d.airline.isin(AIRLINES)&dates.dt.year.between(2024,2026)
    presentation=d.loc[scope].copy()
    dashboard=d.loc[d.airline.isin(AIRLINES)&dates.ge(pd.Timestamp('2024-01-01',tz='UTC'))&~(flag(d.long_haul)|flag(d.first_class))].copy()
    check('Historical presentation count',len(presentation)==EXPECTED_PRESENTATION,f'Actual {len(presentation):,}; historical {EXPECTED_PRESENTATION:,}')
    check('Invalid dates',not dates.isna().any(),f'{dates.isna().sum():,} invalid/missing dates')
    check('Recognized airlines',d.airline.notna().all(),f'{d.airline.isna().sum():,} unmatched airline names')
    for name,frame in [('Presentation',presentation),('Dashboard',dashboard)]:
        absent=set(AIRLINES)-set(frame.airline.dropna())
        check(name+' coverage',not absent,'Missing airlines: '+', '.join(sorted(absent)) if absent else 'All seven airlines present',True)
        labels=frame.gemini_overall_sentiment.astype('string').str.strip().str.lower()
        invalid=int((~labels.isin(VALID)).sum())
        check(name+' Gemini labels',invalid==0,f'{invalid:,} missing/unrecognized overall labels',name=='Presentation')
    sample_summary=pd.DataFrame({'Matched all dates':d.airline.value_counts(),'Presentation 2024–2026':presentation.airline.value_counts(),'Dashboard 2024 onwards':dashboard.airline.value_counts()}).reindex(AIRLINES).fillna(0).astype(int)
    sample_summary.loc['TOTAL']=sample_summary.sum()
    display(sample_summary)
except Exception as exc:
    check('Matched analysis',False,'Unable to calculate: '+str(exc),True)

,Matched all dates,Presentation 2024–2026,Dashboard 2024 onwards
airline,,,
Aegean Airlines,3062,2716,2716
Turkish Airlines,8546,7535,7535
Air France,4741,4101,4101
Lufthansa,10365,8161,8161
Bangkok Air,460,374,374
Porter Airlines,1799,1692,1692
Virgin Australia,1246,989,989
TOTAL,30219,25568,25568


## 4. Final report and recovery list

In [5]:
audit=pd.DataFrame(results)
summary=pd.crosstab(audit.stage,audit.status).reindex(columns=['PASS','MISSING','WARNING','ERROR'],fill_value=0)
display(summary)
issues=audit.loc[audit.status.ne('PASS')].copy()
blockers=issues.loc[issues.required & issues.status.isin(['MISSING','ERROR'])]
print(f'Required failures: {len(blockers)} | Other findings: {len(issues)-len(blockers)}')
print('RESULT:', 'ACTION REQUIRED' if len(blockers) else 'REQUIRED CHECKS PASSED')
with pd.option_context('display.max_rows',None,'display.max_colwidth',None):
    display(issues.sort_values(['required','stage','path'],ascending=[False,True,True]))
if SAVE_REPORT:
    out=ROOT/'Presentation'/'Integrity_Check'; out.mkdir(parents=True,exist_ok=True)
    stamp=datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    audit.to_csv(out/f'project_integrity_{stamp}.csv',index=False)
    issues.to_csv(out/f'project_issues_{stamp}.csv',index=False)
    if not sample_summary.empty: sample_summary.to_csv(out/f'matched_sample_{stamp}.csv')
    print('Reports saved in',out)
else: print('Report export disabled.')

status,PASS,MISSING,WARNING,ERROR
stage,,,,
Cleaning code,1,0,0,0
Cleaning data,4,0,0,0
Dataset contract,4,0,0,0
Documentation,2,0,0,0
Gemini code,1,0,0,0
Gemini data,1,0,0,0
Gemini recovery / diagnostics,7,0,1,0
HF code,3,0,0,0
HF data,2,0,0,0


Required failures: 0 | Other findings: 6
RESULT: REQUIRED CHECKS PASSED


,stage,path,required,status,detail
59,Gemini recovery / diagnostics,Data\Output\LLM\gemini_missing_reviews.csv,False,WARNING,Optional path absent
66,Pilot,Data\Output\LLM\Pilot\gemini_failures.jsonl,False,WARNING,Empty optional log; may indicate no failures
65,Pilot,Data\Output\LLM\Pilot\haiku_failures.jsonl,False,WARNING,Empty optional log; may indicate no failures
120,Repository hygiene,.gitignore,False,WARNING,Optional path absent
121,Repository hygiene,requirements.txt,False,WARNING,Optional path absent
33,Translation recovery,Data\Output\Cleaning data\incremental_translation_checkpoint.csv,False,WARNING,Optional path absent


Reports saved in C:\Users\mfess\GitHub_Repos\aegean airlines\Presentation\Integrity_Check


## Interpretation
Missing optional logs or legacy files do not prevent dashboard execution. Missing production inputs do. A historical sample-count difference can reflect newly collected reviews or different scope. CSV checks do not establish correctness of model predictions, checkpoint compatibility, human annotation quality, or exact agreement with PowerPoint figures. No credentials or raw review text are exported in the audit tables.